# 07 · Niepewność modelu z augmentacji (TTA, grupa D4): wzory, działanie, wizualizacja

**Cel.** Opisać dla każdego jądra, **jak pewnie student je segmentuje**, bez etykiet i bez dropoutu. Te miary będą
wejściami pierwszego selektora rozmytego (bez cech morfologicznych).

**Pomysł.** Pokazujemy studentowi ten sam obraz w 8 wersjach (obroty o 90° i odbicia lustrzane, czyli symetrie kwadratu,
grupa $D_4$). Każdą segmentację przenosimy z powrotem do orientacji oryginału. Te przekształcenia są odwracalne co do
piksela, więc każda różnica między segmentacjami pochodzi z modelu, a nie z przekształcenia. Model pewny narysuje
w każdej wersji to samo jądro; niepewny narysuje je inaczej, podzieli, połączy z sąsiadem albo pominie.

**Oznaczenia.** $P_i$ = piksele jądra $i$ w segmentacji oryginału (jądra odniesienia, $i=1..N$); $Q_{tj}$ = piksele jądra
$j$ w wersji $t$ ($t=1..T$, $T=7$), już przeniesione do układu oryginału; $|X|$ = liczba pikseli.

**Dopasowanie** (zawsze z perspektywy jądra oryginału):
$$\mathrm{IoU}(P_i,Q_{tj})=\frac{|P_i\cap Q_{tj}|}{|P_i\cup Q_{tj}|},\qquad b_t(i)=\max_j \mathrm{IoU}(P_i,Q_{tj}),\qquad m_t(i)=[\,b_t(i)>0{,}5\,]$$
Przy progu $>0{,}5$ dopasowanie jest jednoznaczne (jak w PQ).

**Miary na jądro:**
- wykrywalność: $D(i)=\frac1T\sum_t m_t(i)$ — czy model konsekwentnie widzi jądro,
- niestabilność konturu: $K(i)=1-\dfrac{\sum_t m_t(i)\,b_t(i)}{\sum_t m_t(i)}$ (gdy nigdy nie dopasowane: $K=1$) — jak „pływa” brzeg,
- niestabilność topologii: $\mathrm{Top}(i)=\frac1T\sum_t[\text{podział}_t(i)\lor\text{scalenie}_t(i)]$, gdzie przy pokryciu
  $c(i,j)=|P_i\cap Q_{tj}|/|P_i|$ i progu $\alpha=0{,}2$:
  podział = co najmniej 2 jądra wersji pokrywają każde $\ge\alpha$ jądra $i$; scalenie = jądro wersji, które pokrywa
  najwięcej $P_i$, pokrywa też $\ge\alpha$ innego jądra oryginału,
- niepewność pikseli ($p_t=\sigma(\text{cellprob}_t)$, $t=0$ to oryginał; $B_i$ = pas $\pm2$ px wokół brzegu jądra $i$;
  $H(p)=-p\log_2p-(1-p)\log_2(1-p)$):
$$U_{TTA}(i)=\overline{\mathrm{Var}_t\,p_t(x)}_{x\in B_i},\quad U_H(i)=\overline{H(p_0(x))}_{x\in B_i},\quad U_{MI}(i)=\overline{H(\bar p(x))-\overline{H(p_t(x))}_t}_{x\in B_i}$$

**Na obraz:** jądra dodatkowe $E$ = średnia po wersjach liczby jąder, które prawie nie leżą na żadnym jądrze oryginału
($<\alpha$ ich pikseli), podzielona przez $N$ — prawdopodobnie jądra pominięte w oryginale.

**Ocena (tylko offline):** błąd jądra względem wyroczni $err(i)=1-\max_j\mathrm{IoU}(P_i,\text{jądro wyroczni}_j)$, jądro
„błędne” gdy $\max\mathrm{IoU}\le0{,}5$; na obraz $u=1-PQ(\text{student},\text{wyrocznia})$. Selektor wyroczni nie widzi.

Moduł: `src/histopathology_pnw/uncertainty/tta.py`. Wymaga GPU (8 inferencji studenta na obraz, bez treningu).

In [ ]:
SEED = 0  # seed losowego AL, z którego bierzemy studenta
N_LABELED = 4  # runda: student wytrenowany na tylu obrazach; analizujemy obrazy NIEoznaczone + val
SYMMETRIES = "D4"  # "D4" (8 wersji) albo "D4_SMALL" (4 wersje: tożsamość, 2 odbicia, obrót 180°)
ALPHA = 0.2  # próg pokrycia dla podziału/scalenia/jąder dodatkowych
RING = 2  # px, pas wokół brzegu dla miar pikselowych
MAX_IMAGES = 5  # np. 5 do szybkiego testu; None = wszystkie

In [ ]:
%matplotlib inline
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from skimage.measure import regionprops_table
from skimage.segmentation import find_boundaries
from tqdm.auto import tqdm

from histopathology_pnw.config import Settings
from histopathology_pnw.datasets import monuseg
from histopathology_pnw.device import get_device
from histopathology_pnw.metrics import panoptic_quality
from histopathology_pnw.models.oracle import oracle_masks
from histopathology_pnw.models.student import load_student
from histopathology_pnw.uncertainty import tta

SYMS = tta.D4 if SYMMETRIES == "D4" else tta.D4_SMALL
OUT = Settings().outputs_dir / "tta_uncertainty" / f"seed{SEED}_n{N_LABELED:02d}_{SYMMETRIES}"
OUT.mkdir(parents=True, exist_ok=True)

seed_dir = Settings().outputs_dir / "al_random" / "full" / f"seed{SEED}"
res = pd.read_csv(seed_dir / "results.csv")
row = res[res["n_labeled"] == N_LABELED].iloc[0]
labeled = json.loads(row["labeled_ids"])
ckpt = Path(row["checkpoint"])
if not ckpt.is_file():
    ckpt = seed_dir / f"round_{N_LABELED:02d}" / "models" / "student"

split = monuseg.load_train_val_split()
by_id = monuseg.samples_by_id("train")
unlabeled = [i for i in split["train"] if i not in labeled]
ids = unlabeled + list(split["val"])
if MAX_IMAGES:
    ids = ids[:MAX_IMAGES]
part = {i: "pula" for i in unlabeled} | {i: "val" for i in split["val"]}
samples = [by_id[i] for i in ids]
print(f"student: {ckpt}")
print(f"wersje: {[s.name for s in SYMS]}")
print(f"obrazy: {len(ids)} (pula nieoznaczona {sum(part[i] == 'pula' for i in ids)}, val {sum(part[i] == 'val' for i in ids)})")

## Działanie: 8 wersji obrazu

Każda wersja to ten sam obraz po obrocie lub odbiciu. Student widzi je jako różne obrazy (także inaczej pocięte na
okienka 256×256), ale jądra są te same, więc dobry model powinien dać te same kontury.

In [ ]:
demo = monuseg.read_image(samples[0].image_path)
crop = demo[:300, :300]
fig, axes = plt.subplots(1, len(SYMS), figsize=(3 * len(SYMS), 3.2))
for ax, s in zip(np.atleast_1d(axes), SYMS, strict=True):
    ax.imshow(s.forward(crop))
    ax.set_title(s.name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
fig.savefig(OUT / "symmetries_demo.png", dpi=100, bbox_inches="tight")

## Predykcje i miary

Dla każdego obrazu: 8 (lub 4) inferencji studenta, przeniesienie wyników do orientacji oryginału, miary na jądro.
Predykcje są zapisywane w `predictions/` (ponowne uruchomienie ich nie powtarza; ok. kilkanaście MB na obraz).
Do oceny offline: błąd jądra względem wyroczni i $u$ obrazu.

In [ ]:
device = get_device()
pred_dir = OUT / "predictions"
pred_dir.mkdir(exist_ok=True)
student = None
preds, rows, img_rows = {}, [], []
oracle = dict(zip(ids, oracle_masks(samples), strict=True))
for s in tqdm(samples, desc="TTA"):
    p = pred_dir / f"{s.image_id}.npz"
    if p.is_file():
        pred = tta.TTAPrediction.load(p)
        secs = np.nan
    else:
        if student is None:
            student = load_student(ckpt, device)
        t0 = time.time()
        pred = tta.predict_tta(student, monuseg.read_image(s.image_path), SYMS)
        secs = time.time() - t0
        pred.save(p)
    preds[s.image_id] = pred
    df, st = tta.nucleus_uncertainty(pred, ALPHA, RING)
    ref, o = pred.masks[0], oracle[s.image_id]
    inter = tta.overlap(ref, o)
    a_ref, a_o = inter.sum(axis=1)[1:], inter.sum(axis=0)[1:]
    union = a_ref[:, None] + a_o[None, :] - inter[1:, 1:]
    iou = np.divide(inter[1:, 1:], union, out=np.zeros(union.shape), where=union > 0)
    df["iou_oracle"] = iou.max(axis=1) if iou.shape[1] else 0.0
    cent = pd.DataFrame(regionprops_table(ref, properties=("label", "centroid")))
    df = df.merge(cent.rename(columns={"centroid-0": "cy", "centroid-1": "cx"}), on="label")
    rows.append(df.assign(image_id=s.image_id, part=part[s.image_id]))
    pq = panoptic_quality(o, ref)
    img_rows.append({"image_id": s.image_id, "part": part[s.image_id], "u": 1 - pq["pq"],
                     "n_nuclei": st["n_nuclei"], "E_extra": st["E_extra"], "seconds": secs})

nuc = pd.concat(rows, ignore_index=True)
nuc["err"] = 1 - nuc["iou_oracle"]
nuc["is_error"] = nuc["iou_oracle"] <= 0.5
imgs = pd.DataFrame(img_rows)
nuc.to_csv(OUT / "nuclei_uncertainty.csv", index=False)
imgs.to_csv(OUT / "images.csv", index=False)
MEAS = ["D", "K", "Top", "U_TTA", "U_H", "U_MI"]
print(f"jądra: {len(nuc)}, błędne względem wyroczni: {nuc['is_error'].mean():.1%}, "
      f"średni czas TTA na obraz: {imgs['seconds'].mean():.1f} s")
nuc[MEAS + ["split_rate", "merge_rate"]].describe(percentiles=[0.5, 0.9, 0.99]).round(4)

## Wizualizacja 1: kontury wszystkich wersji

Fragment obrazu z najbardziej niestabilnym jądrem (najwyższe `Top`, potem najniższe `D`). **Zielony**: brzeg
leży w tym samym miejscu we wszystkich wersjach (model stabilny). **Czerwony**: brzeg tylko w części wersji (im
jaśniejszy, tym w większej liczbie); czerwone linie w środku jądra oznaczają, że w niektórych wersjach jądro jest
podzielone, a czerwone mostki między jądrami, że w niektórych wersjach są połączone.

In [ ]:
def contour_agreement(image_crop, masks_crop):
    # zielony = brzeg w każdej wersji, czerwony = brzeg tylko w części wersji (jasność ~ odsetek wersji)
    b = np.stack([find_boundaries(m, mode="inner") for m in masks_crop]).mean(axis=0)
    ov = image_crop.astype(float) / 255
    ov[b == 1] = (0.1, 0.9, 0.1)
    part = (b > 0) & (b < 1)
    ov[part] = np.array([1.0, 0.0, 0.0]) * (0.4 + 0.6 * b[part, None])
    return ov


def worst_nucleus(df):
    return df.sort_values(["Top", "D", "K"], ascending=[False, True, False]).iloc[0]


img_scores = nuc.groupby("image_id")[["Top", "K"]].mean().sum(axis=1)
IID = img_scores.idxmax()
w = worst_nucleus(nuc[nuc["image_id"] == IID])
H = 110
cy, cx = int(w["cy"]), int(w["cx"])
image = monuseg.read_image(by_id[IID].image_path)
y0, x0 = max(cy - H, 0), max(cx - H, 0)
sl = (slice(y0, y0 + 2 * H), slice(x0, x0 + 2 * H))
pred = preds[IID]

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))
axes[0].imshow(contour_agreement(image[sl], [m[sl] for m in pred.masks]))
axes[0].set_title(f"{IID}: zielony = kontur zgodny we wszystkich {len(pred.masks)} wersjach,\nczerwony = kontur tylko w części wersji")
axes[1].imshow(image[sl])
axes[1].contour(pred.masks[0][sl] > 0, levels=[0.5], colors="w", linewidths=0.8)
axes[1].set_title(f"oryginał; jądro w środku: D={w['D']:.2f}, K={w['K']:.2f}, Top={w['Top']:.2f}")
for a in axes:
    a.axis("off")
plt.tight_layout()
fig.savefig(OUT / "contours_all_versions.png", dpi=100, bbox_inches="tight")

## Wizualizacja 2: mapy pikselowe

$\bar p$ = średnie prawdopodobieństwo „piksel należy do jądra” po wersjach; $\mathrm{Var}_t\,p_t$ = rozrzut między
wersjami; $H(\bar p)-\overline{H(p_t)}$ = część niepewności wynikająca z niezgodności wersji (odpowiednik BALD).
Białe kontury: jądra oryginału.

In [ ]:
P = 1 / (1 + np.exp(-np.stack([c[sl] for c in pred.cellprob]).astype(np.float64)))
pbar, var = P.mean(axis=0), P.var(axis=0)
mi = tta.binary_entropy(pbar) - tta.binary_entropy(P).mean(axis=0)
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
for ax, (m, title, cmap) in zip(
    axes,
    [(pbar, "średnie p (po wersjach)", "gray"), (var, "wariancja p między wersjami", "magma"), (mi, "H(p̄) − średnia H(p_t)", "magma")],
    strict=True,
):
    im = ax.imshow(m, cmap=cmap)
    ax.contour(pred.masks[0][sl] > 0, levels=[0.5], colors="w", linewidths=0.6)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.75)
plt.tight_layout()
fig.savefig(OUT / "pixel_maps.png", dpi=100, bbox_inches="tight")

## Wizualizacja 3: jądra pokolorowane miarami

Cały obraz z poprzednich wizualizacji; każde jądro oryginału wypełnione kolorem według wartości miary.
Dla `D` ciemne = rzadko wykrywane; dla pozostałych jasne = bardziej niepewne.

In [ ]:
g = nuc[nuc["image_id"] == IID]
ref = pred.masks[0]


def paint(col):
    lut = np.full(int(ref.max()) + 1, np.nan)
    lut[g["label"].to_numpy()] = g[col].to_numpy()
    return np.where(ref > 0, lut[ref], np.nan)


fig, axes = plt.subplots(2, 3, figsize=(20, 13))
for ax, col in zip(axes.ravel(), MEAS, strict=True):
    ax.imshow(image)
    vals = g[col]
    vmax = 1 if col in ("D", "K", "Top") else np.nanpercentile(vals, 99)
    im = ax.imshow(paint(col), cmap="magma", vmin=0, vmax=vmax, alpha=0.8)
    ax.set_title(col)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
fig.savefig(OUT / "nuclei_colored_by_measure.png", dpi=90, bbox_inches="tight")

## Wizualizacja 4: galeria jąder

Po dwa jądra z całego zbioru: najniższa wykrywalność `D`, najwyższa niestabilność konturu `K` (wśród zawsze
wykrywanych), najwyższa niestabilność topologii `Top`. Kolory konturów jak w wizualizacji 1 (zielony zgodny, czerwony niezgodny).

In [ ]:
picks = pd.concat([
    nuc.nsmallest(2, "D").assign(kat="niska D"),
    nuc[nuc["D"] == 1].nlargest(2, "K").assign(kat="wysokie K"),
    nuc.nlargest(2, "Top").assign(kat="wysokie Top"),
])
h = 40
fig, axes = plt.subplots(1, len(picks), figsize=(3.3 * len(picks), 3.8))
for ax, (_, r) in zip(np.atleast_1d(axes), picks.iterrows(), strict=True):
    im_ = monuseg.read_image(by_id[r["image_id"]].image_path)
    pr = preds[r["image_id"]]
    yy, xx = max(int(r["cy"]) - h, 0), max(int(r["cx"]) - h, 0)
    s2 = (slice(yy, yy + 2 * h), slice(xx, xx + 2 * h))
    ax.imshow(contour_agreement(im_[s2], [m[s2] for m in pr.masks]))
    ax.set_title(f"{r['kat']}\nD={r['D']:.2f} K={r['K']:.2f} Top={r['Top']:.2f}\nIoU z wyrocznią={r['iou_oracle']:.2f}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
fig.savefig(OUT / "gallery.png", dpi=110, bbox_inches="tight")

## Rozkłady miar i ich powiązania

Histogramy z zaznaczonymi percentylami p50 i p90 (kandydaci na punkty zbiorów rozmytych) oraz korelacje Spearmana
między miarami: pary bardzo silnie skorelowane (|ρ| > 0,8) niosą tę samą informację.

In [ ]:
fig, axes = plt.subplots(1, len(MEAS), figsize=(4 * len(MEAS), 3.6))
for ax, col in zip(axes, MEAS, strict=True):
    v = nuc[col].dropna()
    ax.hist(v, bins=40, color="gray")
    for q, ls in ((0.5, "--"), (0.9, ":")):
        ax.axvline(v.quantile(q), color="r", ls=ls, label=f"p{int(q * 100)}={v.quantile(q):.3g}")
    ax.set_yscale("log")
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.tight_layout()
fig.savefig(OUT / "distributions.png", dpi=100, bbox_inches="tight")

corr = nuc[MEAS].corr(method="spearman")
corr.to_csv(OUT / "measures_spearman.csv")
corr.round(2)

## Czy miary wskazują błędy? (analiza offline, z wyrocznią)

Dla każdej miary: korelacja Spearmana z błędem jądra `err` i AUC dla jąder błędnych (IoU z wyrocznią ≤ 0,5).
AUC = prawdopodobieństwo, że losowe jądro błędne ma wyższą wartość miary niż poprawne; `AUC_sila` = max(AUC, 1 − AUC),
`kierunek` ↑ = błąd oznacza wyższe wartości. Dla `D` spodziewamy się ↓ (błędne jądra są rzadziej wykrywane).

In [ ]:
def auc(score, positive):
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = ~np.isnan(score)
    score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(score)
    return float((r[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


tab = []
for col in MEAS + ["split_rate", "merge_rate"]:
    a = auc(nuc[col], nuc["is_error"])
    tab.append({"miara": col, "spearman_z_err": nuc[col].corr(nuc["err"], method="spearman"),
                "AUC": a, "AUC_sila": max(a, 1 - a), "kierunek": "↑" if a >= 0.5 else "↓"})
val_tab = pd.DataFrame(tab).sort_values("AUC_sila", ascending=False)
val_tab.to_csv(OUT / "measures_vs_error.csv", index=False)

fig, axes = plt.subplots(1, len(MEAS), figsize=(4 * len(MEAS), 3.6))
for ax, col in zip(axes, MEAS, strict=True):
    v = nuc[col].dropna()
    bins = np.linspace(v.min(), v.quantile(0.995) if v.quantile(0.995) > v.min() else v.max(), 30)
    for flag, lab in ((False, "poprawne"), (True, "błędne")):
        ax.hist(nuc.loc[nuc["is_error"] == flag, col].dropna(), bins=bins, density=True, histtype="step", lw=1.5, label=lab)
    ax.set_title(col)
axes[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT / "measures_error_vs_ok.png", dpi=100, bbox_inches="tight")
val_tab.round(3)

## Na poziomie obrazu

Proste agregaty miar (średnia po jądrach, średnia z 10% „najgorszych” jąder) i odsetek jąder dodatkowych `E_extra`,
porównane z błędem obrazu $u$ (Spearman po obrazach; przy ~30 obrazach wartości orientacyjne). To wstęp do wyboru
sposobu agregacji w selektorze.

In [ ]:
def top10(x, worst_high=True):
    x = x.dropna().sort_values(ascending=not worst_high)
    k = max(1, int(np.ceil(0.1 * len(x))))
    return x.iloc[:k].mean()


agg_rows = []
for iid, g2 in nuc.groupby("image_id"):
    r = {"image_id": iid}
    for col in MEAS:
        r[f"{col}_sr"] = g2[col].mean()
        r[f"{col}_top10"] = top10(g2[col], worst_high=(col != "D"))
    agg_rows.append(r)
agg = pd.DataFrame(agg_rows).merge(imgs, on="image_id")
agg.to_csv(OUT / "image_aggregates.csv", index=False)
cols = [c for c in agg.columns if c.endswith(("_sr", "_top10"))] + ["E_extra"]
img_corr = pd.Series({c: agg[c].corr(agg["u"], method="spearman") for c in cols}, name="spearman_z_u")
print(f"obrazów: {len(agg)}")
img_corr.sort_values(key=np.abs, ascending=False).round(3)

## Jak czytać i co dalej

- **Kontury wszystkich wersji**: zgodne kontury = stabilny model; rozjeżdżające się, dzielone lub łączone = niepewny.
- **`D`, `K`, `Top`** opisują stabilność wyniku (masek), **`U_TTA`, `U_H`, `U_MI`** niepewność na poziomie pikseli.
  Tabela korelacji pokaże, które niosą tę samą informację (wtedy do systemu bierzemy jedną).
- **Tabela AUC** to główne kryterium wyboru wejść systemu rozmytego: miary z `AUC_sila` wyraźnie powyżej 0,5 wskazują
  jądra źle segmentowane. Uwaga: wyrocznia służy tu tylko do sprawdzenia; parametry zbiorów rozmytych bierzemy
  z rozkładów miar (histogramy, percentyle), bez etykiet.
- **Spodziewane wartości**: Cellpose-SAM był uczony z obrotami i odbiciami, więc większość jąder ma `D = 1` i małe `K`;
  informacja jest w ogonach rozkładów (dlatego histogramy mają skalę logarytmiczną).
- Inne rundy/seedy: zmień `SEED` i `N_LABELED`; `SYMMETRIES = "D4_SMALL"` sprawdza, czy 4 wersje wystarczą.
- Następny krok: zmienne lingwistyczne i baza reguł z tych miar (notebook z systemem rozmytym).